# acai.logging - Test Notebook

Interactive test suite for the hexagonal logging module.  
Run each cell top-to-bottom to verify all components.

**Architecture under test:**

| Layer | Component | Description |
|-------|-----------|-------------|
| Port | `LoggerPort` | Abstract interface every adapter implements |
| Domain | `Logger` | Application service - context stack + Lambda decorator |
| Domain | `LoggerConfig` | Configuration value object |
| Adapter | `ConsoleLogger` | Stdout (text or JSON) |
| Adapter | `CloudWatchLogger` | AWS Lambda PowerTools → CloudWatch |
| Factory | `create_logger()` | Composition root |

## 0 - Setup & Imports

In [6]:
import sys, os, shutil, io, json, tempfile
from pathlib import Path
from unittest.mock import MagicMock, patch
from contextlib import redirect_stderr

# Ensure the acai package is importable
_project_root = Path.cwd()
while _project_root.name != "acai" and _project_root != _project_root.parent:
    _project_root = _project_root.parent
_shared_python = _project_root.parent  # …/shared/python
if str(_shared_python) not in sys.path:
    sys.path.insert(0, str(_shared_python))

from acai.logging import (
    create_logger,
    LoggerPort,
    LogLevel,
    LoggerConfig,
    Logger,
    LoggerContext,
)
from acai.logging.adapters.outbound.console_logger import ConsoleLogger

# Track test results
_results: list[tuple[str, bool, str]] = []

def _record(name: str, passed: bool, detail: str = "") -> None:
    _results.append((name, passed, detail))
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}  {name}" + (f"  - {detail}" if detail else ""))

print("Imports OK ✔")

Imports OK ✔


## 1 - Console Logger (text format)

Basic smoke test: create a `ConsoleLogger` directly and verify all five log levels work.

In [7]:
import logging

# Use a unique logger name per test to avoid handler accumulation
_adapter = ConsoleLogger(logger_name="test1_text", level=LogLevel.DEBUG)

# Capture output via a custom handler
_buf = io.StringIO()
_handler = logging.StreamHandler(_buf)
_handler.setFormatter(logging.Formatter("%(levelname)s|%(message)s"))
logging.getLogger("test1_text").addHandler(_handler)

_adapter.debug("d-msg")
_adapter.info("i-msg")
_adapter.warning("w-msg")
_adapter.error("e-msg")
_adapter.critical("c-msg")

output = _buf.getvalue()
expected_levels = ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]
all_present = all(lvl in output for lvl in expected_levels)
_record("Console text - all levels", all_present, f"captured {len(output)} chars")
print(output)

2026-04-02 14:00:50,900 - DEBUG - d-msg
2026-04-02 14:00:50,901 - INFO - i-msg
2026-04-02 14:00:50,901 - WARNING - w-msg
2026-04-02 14:00:50,902 - ERROR - e-msg
2026-04-02 14:00:50,902 - CRITICAL - c-msg


✅ PASS  Console text - all levels  - captured 64 chars
DEBUG|d-msg
INFO|i-msg
WARNING|w-msg
ERROR|e-msg
CRITICAL|c-msg



## 2 - Console Logger (JSON format)

Verify that setting `json_output=True` produces valid JSON log lines.

In [8]:
from acai.logging.adapters.outbound.console_logger import _JsonFormatter

_json_adapter = ConsoleLogger(logger_name="test2_json", level=LogLevel.DEBUG, json_output=True)

_buf2 = io.StringIO()
_h2 = logging.StreamHandler(_buf2)
# Use the same JSON formatter so we capture structured output
_h2.setFormatter(_JsonFormatter())
logging.getLogger("test2_json").addHandler(_h2)

_json_adapter.info("json-test", user="alice", action="login")

lines = [l for l in _buf2.getvalue().strip().splitlines() if l.strip()]
# The JSON-formatted line is produced by our _JsonFormatter; find it.
parsed = None
for line in lines:
    try:
        parsed = json.loads(line)
        break
    except json.JSONDecodeError:
        continue

valid_json = parsed is not None and parsed.get("message") == "json-test"
_record("Console JSON - valid output", valid_json, f"parsed: {parsed}")
print(json.dumps(parsed, indent=2) if parsed else "NO JSON FOUND")

{"timestamp": "2026-04-02 14:00:50,911", "level": "INFO", "logger": "test2_json", "message": "json-test", "user": "alice", "action": "login"}


✅ PASS  Console JSON - valid output  - parsed: {'timestamp': '2026-04-02 14:00:50,911', 'level': 'INFO', 'logger': 'test2_json', 'message': 'json-test', 'user': 'alice', 'action': 'login'}
{
  "timestamp": "2026-04-02 14:00:50,911",
  "level": "INFO",
  "logger": "test2_json",
  "message": "json-test",
  "user": "alice",
  "action": "login"
}


## 3 - FileLogger via StorageWriter

Create a `FileLogger` backed by `LocalFileStorage` and confirm log lines are persisted.

In [9]:
from acai.logging import LoggerConfig, LogLevel, create_local_logger
from acai.storage import create_storage

_tmpdir = Path(tempfile.mkdtemp())

try:
    # Bootstrap logger used by the storage adapter itself
    _bootstrap = create_logger(
        LoggerConfig(service_name="bootstrap", log_level=LogLevel.WARNING)
    )
    _storage = create_storage(_bootstrap)

    _log_path = _tmpdir / "test3.log"

    _file_logger = create_local_logger(
        LoggerConfig(service_name="test3", log_level=LogLevel.DEBUG),
        storage=_storage,
        log_path=str(_log_path),
    )
    _file_logger.info("written to file", key="value")
    _file_logger.flush()

    file_exists = _log_path.exists()
    content = _storage.read(_log_path) if file_exists else ""
    has_content = "written to file" in content
    _record("File logger - file created", file_exists)
    _record("File logger - content written", has_content, f"{len(content)} bytes")
    print(content)
finally:
    shutil.rmtree(_tmpdir, ignore_errors=True)

✅ PASS  File logger - file created
✅ PASS  File logger - content written  - 57 bytes
2026-04-02 12:00:50 - INFO - written to file | key=value



## 4 - Context Stack (push / pop)

`Logger.push_context()` merges key-value pairs into **every** subsequent log call until `pop_context()` is called.

In [10]:
# Use a mock adapter to capture what Logger sends downstream
_mock = MagicMock(spec=LoggerPort)
_logger = Logger(_mock)

_logger.push_context({"request_id": "r-42"})
_logger.push_context({"user": "alice"})
_logger.info("test-ctx-msg", extra_key="val")

# The mock's log() should have been called with both context keys merged
call_kwargs = _mock.log.call_args
_, kwargs = call_kwargs
passed = (
    kwargs.get("request_id") == "r-42"
    and kwargs.get("user") == "alice"
    and kwargs.get("extra_key") == "val"
)
_record("Context stack - keys merged", passed, f"kwargs={kwargs}")

# Pop one context layer
_logger.pop_context()
_mock.reset_mock()
_logger.info("after-pop")
kwargs2 = _mock.log.call_args[1]
still_has_request = kwargs2.get("request_id") == "r-42"
lost_user = "user" not in kwargs2
_record("Context stack - pop removes top", still_has_request and lost_user, f"kwargs={kwargs2}")

_logger.clear_context()

✅ PASS  Context stack - keys merged  - kwargs={'request_id': 'r-42', 'user': 'alice', 'extra_key': 'val'}
✅ PASS  Context stack - pop removes top  - kwargs={'request_id': 'r-42'}


## 5 - LoggerContext Manager

`LoggerContext` wraps push/pop in a `with` block so context is automatically cleaned up - even on exceptions.

In [11]:
_mock2 = MagicMock(spec=LoggerPort)
_logger2 = Logger(_mock2)

with LoggerContext(_logger2, {"batch": "b-7"}):
    _logger2.info("inside-ctx")
    ctx_inside = _mock2.log.call_args[1]

_mock2.reset_mock()
_logger2.info("outside-ctx")
ctx_outside = _mock2.log.call_args[1]

has_batch_inside = ctx_inside.get("batch") == "b-7"
no_batch_outside = "batch" not in ctx_outside
_record("LoggerContext - key present inside", has_batch_inside)
_record("LoggerContext - key absent after exit", no_batch_outside)

# Verify cleanup on exception
_mock3 = MagicMock(spec=LoggerPort)
_logger3 = Logger(_mock3)
try:
    with LoggerContext(_logger3, {"err_ctx": True}):
        raise ValueError("boom")
except ValueError:
    pass

_record(
    "LoggerContext - cleanup on exception",
    _logger3.get_current_context() == {},
    f"ctx={_logger3.get_current_context()}",
)

✅ PASS  LoggerContext - key present inside
✅ PASS  LoggerContext - key absent after exit
✅ PASS  LoggerContext - cleanup on exception  - ctx={}


## 6 - Dynamic Log-Level Change

After calling `set_level(WARNING)`, debug and info messages should be suppressed.

In [12]:
_buf6 = io.StringIO()
_adapter6 = ConsoleLogger(logger_name="test6_level", level=LogLevel.DEBUG)
_h6 = logging.StreamHandler(_buf6)
_h6.setFormatter(logging.Formatter("%(levelname)s|%(message)s"))
logging.getLogger("test6_level").addHandler(_h6)

_adapter6.info("should-appear")
_adapter6.set_level(LogLevel.WARNING)
_adapter6.debug("should-NOT-appear")
_adapter6.info("should-NOT-appear-either")
_adapter6.warning("should-appear-too")

out6 = _buf6.getvalue()
_record("Dynamic level - info before change", "should-appear" in out6)
_record("Dynamic level - debug suppressed", "should-NOT-appear" not in out6.split("should-appear-too")[0].split("should-appear")[-1])
_record("Dynamic level - warning still works", "should-appear-too" in out6)
print(out6)

2026-04-02 14:00:50,959 - INFO - should-appear
2026-04-02 14:00:50,960 - WARNING - should-appear-too


✅ PASS  Dynamic level - info before change
✅ PASS  Dynamic level - debug suppressed
✅ PASS  Dynamic level - warning still works
INFO|should-appear
WARNING|should-appear-too



## 7 - CloudWatch Logger (mocked)

We mock `aws_lambda_powertools.Logger` so the test runs anywhere - no AWS credentials needed.

In [13]:
import importlib

# Mock aws_lambda_powertools before importing the adapter
_mock_pt = MagicMock()
_mock_pt_logger_instance = MagicMock()
_mock_pt.Logger.return_value = _mock_pt_logger_instance

with patch.dict("sys.modules", {"aws_lambda_powertools": _mock_pt}):
    # Force reimport so patched module is used
    import acai.logging.adapters.outbound.cloudwatch_logger as _cw_mod
    importlib.reload(_cw_mod)
    CloudWatchLogger = _cw_mod.CloudWatchLogger

    cw = CloudWatchLogger(service="test-svc", level=LogLevel.INFO)
    cw.info("cw-test-msg", doc_id="SR-210")

    # Verify the powertools Logger.log() was called
    called = _mock_pt_logger_instance.log.called
    call_args = _mock_pt_logger_instance.log.call_args
    _record("CloudWatch - adapter calls powertools", called, f"args={call_args}")

    # set_level now goes through _logger._logger.setLevel for PowerTools
    level_set = _mock_pt_logger_instance._logger.setLevel.called
    _record("CloudWatch - level set on init", level_set)

✅ PASS  CloudWatch - adapter calls powertools  - args=call(20, 'cw-test-msg')
✅ PASS  CloudWatch - level set on init


## 8 - `inject_lambda_context` Decorator

Simulate a Lambda invocation and verify the decorator enriches logs with request ID, cold-start flag, and function name.

In [14]:
_mock8 = MagicMock(spec=LoggerPort)
_logger8 = Logger(_mock8)

class FakeLambdaContext:
    aws_request_id = "req-abc-123"
    function_name = "my-func"
    function_version = "$LATEST"
    memory_limit_in_mb = 128
    log_group_name = "/aws/lambda/my-func"
    log_stream_name = "2026/03/23/[$LATEST]xyz"
    @staticmethod
    def get_remaining_time_in_millis():
        return 300_000

@_logger8.inject_lambda_context(
    include_event=True,
    include_context=True,
    include_cold_start=True,
)
def fake_handler(event, context):
    _logger8.info("inside-handler")
    return {"status": "ok"}

result = fake_handler({"action": "embed"}, FakeLambdaContext())

# Collect all calls to the mock adapter
all_calls = _mock8.log.call_args_list
all_kwargs = [c[1] for c in all_calls]

# Check that at least one call includes the request_id
has_request_id = any(kw.get("aws_request_id") == "req-abc-123" for kw in all_kwargs)
has_cold_start = any(kw.get("cold_start") is True for kw in all_kwargs)
handler_returned = result == {"status": "ok"}

_record("Lambda decorator - request_id injected", has_request_id)
_record("Lambda decorator - cold_start tracked", has_cold_start)
_record("Lambda decorator - handler return value", handler_returned, f"result={result}")

# Second invocation should show cold_start=False
_mock8.reset_mock()
fake_handler({"action": "embed2"}, FakeLambdaContext())
all_calls2 = _mock8.log.call_args_list
all_kwargs2 = [c[1] for c in all_calls2]
cold_start_false = any(kw.get("cold_start") is False for kw in all_kwargs2)
_record("Lambda decorator - cold_start=False on 2nd call", cold_start_false)

✅ PASS  Lambda decorator - request_id injected
✅ PASS  Lambda decorator - cold_start tracked
✅ PASS  Lambda decorator - handler return value  - result={'status': 'ok'}
✅ PASS  Lambda decorator - cold_start=False on 2nd call


## 9 - `create_logger()` Factory

Test the factory with default config and a custom `LoggerConfig`.

In [15]:
import importlib
import acai.logging.adapters.outbound.cloudwatch_logger as _cw_mod
from acai.logging import create_lambda_logger

# Default factory - should produce a Logger backed by ConsoleLogger
default_logger = create_logger()
is_logger = isinstance(default_logger, Logger)
_record("Factory - returns Logger", is_logger)

# Custom config
cfg = LoggerConfig(
    service_name="custom-svc",
    log_level=LogLevel.WARNING,
    json_output=True,
)
custom_logger = create_logger(cfg)
_record("Factory - custom config accepted", isinstance(custom_logger, Logger))

# Factory with create_lambda_logger (mocked)
with patch.dict("sys.modules", {"aws_lambda_powertools": _mock_pt}):
    importlib.reload(_cw_mod)
    cw_logger = create_lambda_logger(cfg)
    _record("Factory - cloudwatch adapter", isinstance(cw_logger, Logger))

✅ PASS  Factory - returns Logger
✅ PASS  Factory - custom config accepted
✅ PASS  Factory - cloudwatch adapter


## 10 - Error Resilience

If the underlying adapter raises, `Logger` should print a fallback message instead of crashing.

In [16]:
_exploding_mock = MagicMock(spec=LoggerPort)
_exploding_mock.log.side_effect = RuntimeError("adapter exploded")

_bomb_logger = Logger(_exploding_mock)

# Logger.log() now falls back to logging.getLogger(__name__).warning()
# Capture it via a custom handler on that logger
_fallback_buf = io.StringIO()
_fallback_handler = logging.StreamHandler(_fallback_buf)
_fallback_handler.setFormatter(logging.Formatter("%(message)s"))
_domain_logger = logging.getLogger("acai.logging.domain.logger")
_domain_logger.addHandler(_fallback_handler)
_domain_logger.setLevel(logging.DEBUG)

try:
    _bomb_logger.info("this should not crash")
    did_not_crash = True
except Exception:
    did_not_crash = False
finally:
    _domain_logger.removeHandler(_fallback_handler)

fallback_output = _fallback_buf.getvalue()
_record("Error resilience - no crash", did_not_crash)
_record(
    "Error resilience - fallback logged",
    "Logger failed" in fallback_output,
    f"output: {fallback_output.strip()!r}",
)

✅ PASS  Error resilience - no crash
✅ PASS  Error resilience - fallback logged  - output: 'Logger failed: adapter exploded. Message: this should not crash'


## Summary

In [17]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total = len(_results)

print("=" * 60)
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print("=" * 60)

if failed:
    print("\nFailed tests:")
    for name, ok, detail in _results:
        if not ok:
            print(f"  ❌ {name}  - {detail}")
else:
    print("\n🎉 All tests passed!")

  RESULTS: 23/23 passed, 0 failed

🎉 All tests passed!
